# 🔄 Incremental Micro-Batch Fact Pipeline (`fact_orders` & `sb_fact_orders`)
This notebook executes the production incremental micro-batch pipeline for **Sales Orders Facts**:
* **Bronze:** Ingests newly arrived incremental CSV orders with explicit schemas into master `bronze.orders` and transient `bronze.staging_orders`, then archives files to processed storage.
* **Silver:** Cleanses, standardizes dates, broadcasts product dimensions, and merges into `silver.orders` and `silver.staging_orders`.
* **Gold Subsidiary (`sb_fact_orders`):** Merges daily incremental sales orders at transaction grain.
* **Gold Enterprise (`fact_orders`):** Identifies touched calendar months, pulls complete month history from subsidiary facts, recomputes entire affected months to avoid partial sums, atomically merges into `fact_orders`, drops staging tables, and runs Z-ORDER optimization.

### 📌 Step 1: Import Core PySpark & Delta Lake Libraries
* **Purpose:** Loads necessary PySpark SQL analytical functions and Delta Lake table abstractions required for micro-batch transactional processing and Lakehouse ACID merges.
* **Logic & Transformations:** Imports `pyspark.sql.functions as F` and `DeltaTable` from `delta.tables`.
* **Inputs & Dependencies:** PySpark runtime and `delta-spark` package.
* **Outputs & Medallion State:** Module namespaces `F` and `DeltaTable` available in session scope.

In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### 📌 Step 2: Runtime Bootstrap & Project Utilities Execution
* **Purpose:** Configures modular Python paths and executes shared project utilities to establish active environment configurations, conformed schemas, and audit tools.
* **Logic & Transformations:** Resolves repository root path on `sys.path`, executes `%run ./utilities`, and initializes Databricks compatibility shims.
* **Inputs & Dependencies:** Shared Lakehouse utilities (`./utilities.py` / `utilities.ipynb`).
* **Outputs & Medallion State:** Pre-populated `spark`, `dbutils`, `display`, and global configurations in session scope.

In [2]:
# Initialize environment & Databricks compatibility (noop in Databricks)
import sys, os
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, "..")) if os.path.basename(current_dir) in ["1_setup", "2_dimension_data_processing", "3_fact_dat_processing"] else current_dir
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.compat import init_notebook_context
spark, dbutils, display = init_notebook_context(globals())

# Load environment config, schemas, and utilities via relative path
%run ../1_setup/utilities


26/09/17 12:45:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### 📌 Step 3: Verify Active Medallion Schema Configurations
* **Purpose:** Confirms that the target Lakehouse schemas (`bronze`, `silver`, `gold`) are properly defined and aligned with the active environment.
* **Logic & Transformations:** Prints `bronze_schema`, `silver_schema`, and `gold_schema` strings to standard output for visual verification.
* **Inputs & Dependencies:** Configuration variables exported by utilities in Step 2.
* **Outputs & Medallion State:** Schema names printed to cell output.

In [3]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


### 📌 Step 4: Pipeline Parameterization & Storage URI Construction
* **Purpose:** Establishes configurable parameters (`catalog`, `data_source` = `orders`) and derives storage URIs for the raw orders landing zone and archive directories.
* **Logic & Transformations:** Defines interactive Databricks text widgets, resolves active catalog and dataset name, and forms S3 paths (`base_path`, `landing_path`, `processed_path`) and table names (`bronze_table`, `silver_table`, `gold_table`).
* **Inputs & Dependencies:** Databricks widget inputs (`catalog`: `fmcg`, `data_source`: `orders`).
* **Outputs & Medallion State:** Pipeline URI paths and target table names registered.

In [4]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://spartsbar-2355/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Base Path:  s3://spartsbar-2355/orders
Landing Path:  s3://spartsbar-2355/orders/landing/
Processed Path:  s3://spartsbar-2355/orders/processed/


### 📌 Step 5: Schema-Enforced Incremental Ingestion from S3 Landing Zone
* **Purpose:** Ingests freshly arrived incremental micro-batch CSV orders files using explicit schema enforcement, capturing ingestion audit metadata.
* **Logic & Transformations:** Binds explicit `orders_schema`, appends `current_timestamp()` as `read_timestamp`, and unpacks `_metadata.file_name` and `_metadata.file_size`.
* **Inputs & Dependencies:** Incremental CSV files landed at `landing_path`.
* **Outputs & Medallion State:** Raw DataFrame `df` containing incremental order records with ingestion metadata.

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(orders_schema)  # Explicit StructType schema eliminates inferSchema scan
        .load(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)
print("Total Arrived Rows: ", df.count())
df.show(5)
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("append").saveAsTable(bronze_table)
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")


26/09/17 12:45:25 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3://spartsbar-2355/orders/landing//*.csv.
org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3586)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3617)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3721)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3672)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:558)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:373)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:57)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSou

[Local Spark Emulation] S3 path detected without AWS credentials. Providing mock data for: s3://spartsbar-2355/orders/landing//*.csv


Total Arrived Rows:  4
+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+
|order_id|order_placement_date|customer_id|product_id|order_qty|           _metadata|      read_timestamp|      file_name|file_size|
+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+
|  ORD001|Tuesday, July 01,...|       1001|      P101|       50|{sample_data.csv,...|2026-09-17 12:45:...|sample_data.csv|     1024|
|  ORD002|          2025-07-15|       1002|      P102|       20|{sample_data.csv,...|2026-09-17 12:45:...|sample_data.csv|     1024|
|  ORD003|          01/08/2025|       1003|      P103|      100|{sample_data.csv,...|2026-09-17 12:45:...|sample_data.csv|     1024|
|  ORD004|     August 20, 2025|       1001|      P104|       15|{sample_data.csv,...|2026-09-17 12:45:...|sample_data.csv|     1024|
+--------+--------------------+-----------+---

26/09/17 12:45:28 ERROR Utils: Aborting task
org.apache.spark.sql.delta.DeltaAnalysisException: [DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION] Cannot create table ('`bronze`.`orders`'). The associated location ('file:/Users/nithin/Projects/Atlikon_DE/spark-warehouse/bronze.db/orders') is not empty and also not a Delta table.
	at org.apache.spark.sql.delta.DeltaErrorsBase.createTableWithNonEmptyLocation(DeltaErrors.scala:3386)
	at org.apache.spark.sql.delta.DeltaErrorsBase.createTableWithNonEmptyLocation$(DeltaErrors.scala:3385)
	at org.apache.spark.sql.delta.DeltaErrors$.createTableWithNonEmptyLocation(DeltaErrors.scala:4266)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.assertPathEmpty(CreateDeltaTableCommand.scala:566)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.checkPathEmpty$1(CreateDeltaTableCommand.scala:187)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.$anonfun$handleCommit$1(CreateDeltaTableCommand.scala:205)
	at org.apache

### 📌 Step 6: Select Raw Ingested Columns
* **Purpose:** Prepares raw incremental DataFrame for persistence by selecting target source columns.
* **Logic & Transformations:** Imports `col` and projects source columns from `df`.
* **Inputs & Dependencies:** Raw DataFrame `df`.
* **Outputs & Medallion State:** DataFrame `df` prepared for dual-write persistence.

In [0]:
from pyspark.sql.functions import col

df = df.withColumn(
    "order_qty",
    col("order_qty").cast("int")
)

### 📌 Step 7: Append Incremental Batch into Bronze Delta Table
* **Purpose:** Appends newly landed raw orders to the persistent, immutable Bronze historical table with Change Data Feed enabled.
* **Logic & Transformations:** Writes `df` with `format('delta')`, sets `delta.enableChangeDataFeed = true`, and saves in `append` mode to `{bronze_table}`.
* **Inputs & Dependencies:** Ingested DataFrame `df`.
* **Outputs & Medallion State:** Master Bronze Delta table `fmcg.bronze.orders` appended with new records.

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

### 📌 Step 8: Materialize Transient Bronze Staging Table (`staging_orders`)
* **Purpose:** Overwrites a dedicated, lightweight staging table with ONLY the current micro-batch to isolate downstream transformations from the massive historical table.
* **Logic & Transformations:** Writes `df` in `overwrite` mode to `{catalog}.{bronze_schema}.staging_{data_source}`.
* **Inputs & Dependencies:** Incremental DataFrame `df`.
* **Outputs & Medallion State:** Transient table `fmcg.bronze.staging_orders` holding current batch only.

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

### 📌 Step 9: File Archival: Move Landed Files to Processed Directory
* **Purpose:** Archives ingested incremental CSV files from the landing directory to the processed directory, preventing duplicate processing.
* **Logic & Transformations:** Lists landed files via `dbutils.fs.ls(landing_path)` and moves each file to `processed_path` using `dbutils.fs.mv()`.
* **Inputs & Dependencies:** Files at `landing_path`.
* **Outputs & Medallion State:** Landing directory cleared; incremental files archived to `processed_path`.

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

### 📌 Step 10: Query Bronze Staging Table to Initialize Silver Cleansing
* **Purpose:** Reads isolated micro-batch records from `staging_orders` to execute cleansing transformations on new data only.
* **Logic & Transformations:** Executes `spark.sql(SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source})`.
* **Inputs & Dependencies:** Transient table `fmcg.bronze.staging_orders`.
* **Outputs & Medallion State:** DataFrame `df_orders` loaded into session scope.

In [0]:
df_orders = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source};")
df_orders.show(2)

### 📌 Step 11: Multi-Format Date Normalization & Sentinel Cleansing
* **Purpose:** Sanitizes incremental transactional fields: filters non-positive quantities, strips weekday text prefixes, parses multi-format dates, and removes duplicate orders.
* **Logic & Transformations:**
  1. Filters `order_qty.isNotNull() & (order_qty > 0) & order_id.isNotNull()`.
  2. Strips leading weekday prefixes via `regexp_replace(order_placement_date, '^[A-Za-z]+,\\s*', '')`.
  3. Parses multi-format date strings (`yyyy/MM/dd`, `dd-MM-yyyy`, `dd/MM/yyyy`, `MMMM dd, yyyy`) into ISO DateType via `coalesce()`.
  4. Applies `dropDuplicates(['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])`.
* **Inputs & Dependencies:** Staging DataFrame `df_orders`.
* **Outputs & Medallion State:** Cleaned DataFrame `df_orders` with standardized dates and positive order quantities.

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

### 📌 Step 12: Inspect Batch Date Boundaries
* **Purpose:** Identifies minimum and maximum transaction dates present in the current incremental batch.
* **Logic & Transformations:** Calculates `min('order_placement_date')` and `max('order_placement_date')`.
* **Inputs & Dependencies:** Cleaned DataFrame `df_orders`.
* **Outputs & Medallion State:** Minimum and maximum dates displayed.

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

### 📌 Step 13: Broadcast Join with Product Master to Attach Surrogate Key
* **Purpose:** Enriches incremental orders by joining with `fmcg.silver.products` on natural key `product_id` to resolve the conformed surrogate key `product_code`.
* **Logic & Transformations:** Performs an optimized broadcast join `df_orders.join(broadcast(df_products), 'product_id', 'left')`.
* **Inputs & Dependencies:** Orders DataFrame `df_orders` and Silver products table `fmcg.silver.products`.
* **Outputs & Medallion State:** Enriched DataFrame `df_joined` containing `product_code` alongside order details.

In [0]:
# Broadcast small dimension table to prevent cluster-wide shuffle join
from pyspark.sql.functions import broadcast
df_products = spark.table(f"{catalog}.{silver_schema}.products")
df_joined = df_orders.join(broadcast(df_products), on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])
df_joined.show(5)


### 📌 Step 14: Merge Incremental Orders into Historical Silver Table
* **Purpose:** Upserts cleansed incremental orders into the permanent Silver historical table using an ACID merge on composite natural keys.
* **Logic & Transformations:** Merges on `silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id`.
* **Inputs & Dependencies:** Enriched DataFrame `df_joined`.
* **Outputs & Medallion State:** Master Silver Delta table `fmcg.silver.orders` updated.

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### 📌 Step 15: Materialize Transient Silver Staging Table
* **Purpose:** Persists cleansed incremental orders into a transient Silver staging table for downstream Gold fact preparation.
* **Logic & Transformations:** Writes `df_joined` in `overwrite` mode to `{catalog}.{silver_schema}.staging_{data_source}`.
* **Inputs & Dependencies:** Enriched DataFrame `df_joined`.
* **Outputs & Medallion State:** Transient table `fmcg.silver.staging_orders` committed on storage.

In [0]:
# stagging for incremental data

df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

### 📌 Step 16: Project Daily Transactional Attributes for Gold Layer
* **Purpose:** Extracts core transactional fields and standardizes column aliases (`order_placement_date -> date`, `customer_id -> customer_code`, `order_qty -> sold_quantity`).
* **Logic & Transformations:** Executes SQL projection against `{catalog}.{silver_schema}.staging_{data_source}`.
* **Inputs & Dependencies:** Transient Silver staging table `fmcg.silver.staging_orders`.
* **Outputs & Medallion State:** Daily Gold DataFrame `df_gold` ready for subsidiary merge.

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {catalog}.{silver_schema}.staging_{data_source};")

df_gold.show(2)

### 📌 Step 17: Count Incremental Daily Orders
* **Purpose:** Confirms total daily record volume in the current incremental batch.
* **Logic & Transformations:** Calls `df_gold.count()`.
* **Inputs & Dependencies:** DataFrame `df_gold`.
* **Outputs & Medallion State:** Batch row count printed to cell output.

In [0]:
df_gold.count()

### 📌 Step 18: Merge Daily Orders into Subsidiary Gold Fact Table (`sb_fact_orders`)
* **Purpose:** Upserts daily subsidiary sales orders into `fmcg.gold.sb_fact_orders` at daily transaction grain.
* **Logic & Transformations:** Merges on composite key `(date, order_id, product_code, customer_code)`.
* **Inputs & Dependencies:** Daily Gold DataFrame `df_gold`.
* **Outputs & Medallion State:** Gold Delta table `fmcg.gold.sb_fact_orders` updated with current batch transactions.

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### 📌 Step 19: Inspect Incremental Transaction Dates
* **Purpose:** Extracts order placement dates from the current batch to prepare for affected month identification.
* **Logic & Transformations:** Queries `SELECT order_placement_date as date FROM {catalog}.{silver_schema}.staging_{data_source}`.
* **Inputs & Dependencies:** Silver staging table `fmcg.silver.staging_orders`.
* **Outputs & Medallion State:** DataFrame `df_child` loaded into session.

In [0]:
# df_child = your incremental daily rows

df_child =  spark.sql(f"SELECT order_placement_date as date FROM {catalog}.{silver_schema}.staging_{data_source}")

incremental_month_df = df_child.select(
    F.trunc("date", "MM").alias("start_month")
).distinct()

incremental_month_df.show()

incremental_month_df.createOrReplaceTempView("incremental_months")

### 📌 Step 20: Identify Affected Calendar Months & Pull All Existing Daily Rows
* **Purpose:** Core Business Logic: To prevent partial monthly sums, this step identifies which calendar months received new orders, creates a temporary view `incremental_months`, and pulls ALL existing daily records for those touched months from `sb_fact_orders`.
* **Logic & Transformations:**
  1. Truncates batch dates to `trunc(date, 'MM') as start_month`.
  2. Creates temporary view `incremental_months`.
  3. Inner-joins `fmcg.gold.sb_fact_orders` with `incremental_months` on month-start to retrieve the COMPLETE daily history for every touched month.
* **Inputs & Dependencies:** Current batch dates and subsidiary table `fmcg.gold.sb_fact_orders`.
* **Outputs & Medallion State:** DataFrame `monthly_table` containing all daily records across all affected calendar months.

In [0]:
monthly_table = spark.sql(f"""
    SELECT date, product_code, customer_code, sold_quantity
    FROM {catalog}.{gold_schema}.sb_fact_orders sbf
    INNER JOIN incremental_months m
        ON trunc(sbf.date, 'MM') = m.start_month
""")

print("Total Rows: ", monthly_table.count())
monthly_table.show(10)

### 📌 Step 21: Verify Affected Calendar Months
* **Purpose:** Lists the distinct calendar months that will be recomputed.
* **Logic & Transformations:** Queries `monthly_table.select('date').distinct().orderBy('date').show()`.
* **Inputs & Dependencies:** DataFrame `monthly_table`.
* **Outputs & Medallion State:** List of recomputed calendar months displayed in output.

In [0]:
monthly_table.select('date').distinct().orderBy('date').show()

### 📌 Step 22: Recompute Monthly Aggregates Across Full Touched Months
* **Purpose:** Re-aggregates full monthly totals for all product-customer combinations across the touched calendar months, guaranteeing 100% accurate totals without partial sums.
* **Logic & Transformations:**
  1. Truncates date to month-start: `month_start = F.trunc('date', 'MM')`.
  2. Groups by `(month_start, product_code, customer_code)`.
  3. Re-sums `sold_quantity = F.sum('sold_quantity')`.
  4. Renames `month_start` back to `date`.
* **Inputs & Dependencies:** Full monthly history DataFrame `monthly_table`.
* **Outputs & Medallion State:** Recomputed monthly DataFrame `df_monthly_recalc` at grain `(date, product_code, customer_code)`.

In [0]:
df_monthly_recalc = (
    monthly_table
    .withColumn("month_start", F.trunc("date", "MM"))
    .groupBy("month_start", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("month_start", "date")   # month_start → date = first of month
)

df_monthly_recalc.show(10, truncate=False)

### 📌 Step 23: Count Recomputed Monthly Records
* **Purpose:** Verifies total monthly summary records produced for the affected calendar periods.
* **Logic & Transformations:** Calls `df_monthly_recalc.count()`.
* **Inputs & Dependencies:** Recomputed DataFrame `df_monthly_recalc`.
* **Outputs & Medallion State:** Recomputed monthly row count printed to output.

In [0]:
df_monthly_recalc.count()

### 📌 Step 24: Data Quality Validation & Parent Fact Merge (`fact_orders`)
* **Purpose:** Validates monthly fact quality rules and atomically updates the enterprise parent table `fmcg.gold.fact_orders` with the recomputed monthly totals.
* **Logic & Transformations:**
  1. Runs quality check: `sold_quantity >= 0` via `run_quality_checks()`.
  2. Merges on composite primary key: `parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code`.
  3. Atomically overwrites existing monthly figures and inserts new product-customer monthly combinations.
* **Inputs & Dependencies:** Recomputed DataFrame `df_monthly_recalc` and target Delta table `fmcg.gold.fact_orders`.
* **Outputs & Medallion State:** Enterprise parent fact table `fmcg.gold.fact_orders` updated with accurate monthly sales.

In [0]:
# Data Quality verification before recomputed month merge
run_quality_checks(df_monthly_recalc, {"sold_qty_positive": ("sold_quantity >= 0", True)}, "fact_orders")

gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(
    df_monthly_recalc.alias("child_gold"),
    "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("Successfully merged recomputed months into parent fact_orders")


### 📌 Step 25: SQL Cleanup: Drop Transient Bronze Staging Table
* **Purpose:** Drops the transient Bronze staging table to maintain a clean Lakehouse metastore and reclaim storage.
* **Logic & Transformations:** Executes `%sql DROP TABLE fmcg.bronze.staging_orders;`.
* **Inputs & Dependencies:** Staging table `fmcg.bronze.staging_orders`.
* **Outputs & Medallion State:** Transient Bronze staging table dropped.

In [0]:
%sql
DROP TABLE fmcg.bronze.staging_orders;

### 📌 Step 26: SQL Cleanup: Drop Transient Silver Staging Table
* **Purpose:** Drops the transient Silver staging table following successful fact rollup.
* **Logic & Transformations:** Executes `%sql DROP TABLE fmcg.silver.staging_orders;`.
* **Inputs & Dependencies:** Staging table `fmcg.silver.staging_orders`.
* **Outputs & Medallion State:** Transient Silver staging table dropped.

In [0]:
%sql
DROP TABLE fmcg.silver.staging_orders;

### 📌 Step 27: Lakehouse Maintenance: File Compaction & Multi-Dimensional Z-Ordering
* **Purpose:** Compacts small Parquet files generated by incremental writes and co-locates data on storage by primary query filter columns (`date`, `product_code`, `customer_code`) for maximum BI query efficiency.
* **Logic & Transformations:** Executes `OPTIMIZE {catalog}.{gold_schema}.fact_orders ZORDER BY (date, product_code, customer_code)`.
* **Inputs & Dependencies:** Target Delta table `fmcg.gold.fact_orders`.
* **Outputs & Medallion State:** Optimized Delta storage layout with Z-Order indexing applied.

In [0]:
# Compaction & Z-ORDER optimization by primary query filter columns
spark.sql(f"OPTIMIZE {catalog}.{gold_schema}.fact_orders ZORDER BY (date, product_code, customer_code)")
